# LabCleaning_160004613
Se carga el dataset `dirty_cafe_sales.csv` y se aplica limpieza de datos para faltantes, valores atípicos, verificación de duplicados e imputaciones solicitadas.

In [1]:
import numpy as np
import pandas as pd

ruta = 'dirty_cafe_sales.csv'
df_raw = pd.read_csv(ruta)
df_raw.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Técnica 1 de faltantes: conteo absoluto de valores nulos por columna.

In [2]:
df_tmp = df_raw.replace(['ERROR', 'UNKNOWN', 'None'], np.nan)
df_tmp.isna().sum().sort_values(ascending=False)

Location            3961
Payment Method      3178
Item                 969
Price Per Unit       533
Total Spent          502
Quantity             479
Transaction Date     460
Transaction ID         0
dtype: int64

Técnica 2 de faltantes: porcentaje de valores nulos por columna.

In [3]:
(df_tmp.isna().mean() * 100).sort_values(ascending=False).round(2)

Location            39.61
Payment Method      31.78
Item                 9.69
Price Per Unit       5.33
Total Spent          5.02
Quantity             4.79
Transaction Date     4.60
Transaction ID       0.00
dtype: float64

In [4]:
df = df_raw.replace(['ERROR', 'UNKNOWN', 'None'], np.nan).copy()

for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

precios_por_item = df.groupby('Item', dropna=False)['Price Per Unit'].nunique(dropna=True).sort_values(ascending=False)
precios_por_item

Item
NaN         6
Cake        1
Coffee      1
Juice       1
Cookie      1
Salad       1
Sandwich    1
Smoothie    1
Tea         1
Name: Price Per Unit, dtype: int64

In [5]:
duplicados_txn = df['Transaction ID'].duplicated().sum()
duplicados_fila = df.duplicated().sum()
columnas_irrelevantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
pd.Series({
    'Duplicados Transaction ID': duplicados_txn,
    'Duplicados de fila completa': duplicados_fila,
    'Columnas irrelevantes (nunique<=1)': str(columnas_irrelevantes)
})

Duplicados Transaction ID              0
Duplicados de fila completa            0
Columnas irrelevantes (nunique<=1)    []
dtype: object

In [6]:
media_total = df['Total Spent'].mean()
std_total = df['Total Spent'].std()
limite_inf = media_total - 3 * std_total
limite_sup = media_total + 3 * std_total
outliers_total_spent = df[(df['Total Spent'] < limite_inf) | (df['Total Spent'] > limite_sup)]
pd.Series({
    'Media Total Spent': round(media_total, 4),
    'Desviación estándar': round(std_total, 4),
    'Límite inferior': round(limite_inf, 4),
    'Límite superior': round(limite_sup, 4),
    'Cantidad outliers': len(outliers_total_spent)
})

Media Total Spent       8.9309
Desviación estándar     6.0045
Límite inferior        -9.0825
Límite superior        26.9443
Cantidad outliers       0.0000
dtype: float64

In [7]:
df_central = df.copy()
for col in ['Payment Method', 'Location']:
    moda_col = df_central[col].mode(dropna=True)
    moda = moda_col.iloc[0] if not moda_col.empty else np.nan
    df_central[col] = df_central[col].fillna(moda)

df_random = df.copy()
rng = np.random.default_rng(42)
for col in ['Payment Method', 'Location']:
    mascara = df_random[col].isna()
    valores = df_random.loc[~mascara, col].to_numpy()
    if mascara.sum() > 0 and len(valores) > 0:
        df_random.loc[mascara, col] = rng.choice(valores, size=mascara.sum(), replace=True)

df_central[['Payment Method', 'Location']].isna().sum().rename('Nulos tras imputación central'), df_random[['Payment Method', 'Location']].isna().sum().rename('Nulos tras imputación aleatoria')

(Payment Method    0
 Location          0
 Name: Nulos tras imputación central, dtype: int64,
 Payment Method    0
 Location          0
 Name: Nulos tras imputación aleatoria, dtype: int64)

In [8]:
df_final = df_central.copy()
df_final['Transaction Date'] = pd.to_datetime(df_final['Transaction Date'], errors='coerce')
date_num = df_final['Transaction Date'].astype('int64').where(df_final['Transaction Date'].notna(), np.nan)
date_num = date_num.interpolate(method='linear', limit_direction='both')
df_final['Transaction Date'] = pd.to_datetime(date_num)

df_final[['Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']].isna().sum()

Quantity            38
Price Per Unit      38
Total Spent         40
Payment Method       0
Location             0
Transaction Date     0
dtype: int64

In [9]:
df_final.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
